# Capstone EDA — Supervised-Learning-Readiness Notebook
## GA 5800 — Summer 2026

**Student:** *Matthew Kavanagh*  
**Dataset:** *GIS Department*  
**Date submitted:** *2026-05-18*

---

This notebook is your scaffold for the 10-stage capstone. Each stage has:
- a markdown cell describing what to produce,
- empty code cells for your analysis,
- a markdown cell for your written interpretation.

You are **not** limited to the cells provided. Add code, markdown, and plots as needed.

### Before you begin
- Read the full assignment brief: `capstone_eda_supervised_readiness.pdf`.
- Set your random seed in the setup cell below.
- Treat the Stage 10 Data Card as a *living* document — fill it in as you complete each stage, not at the end.

### Submission reminders
- Every p-value must be paired with an **effect size**.
- Every plot must have a title, axis labels (with units), and a caption stating the takeaway.
- Notebook must execute top-to-bottom on a fresh kernel.
- Submit alongside `<lastname>_<firstname>_data_card.md` and `<lastname>_<firstname>_readiness_memo.pdf` as a single zipped folder.

---

## Setup

Imports, random seed, and data load.

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from datetime import datetime
import missingno as msno
import pingouin as pg


# Stats
from scipy import stats
from scipy.stats import (
    chi2_contingency, kruskal, mannwhitneyu, f_oneway,
    pointbiserialr, spearmanr, kendalltau, pearsonr,
)
from scipy.stats import skew, kurtosis

# sklearn
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer

# Multicollinearity (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.distributions.empirical_distribution import ECDF

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display + plot defaults
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
# Load the dataset
ds1 = pd.read_csv('dataset_address/Address_Datasets/dataset1.csv')
ds2 = pd.read_csv('dataset_address/Address_Datasets/dataset2.csv')
ds3 = pd.read_csv('dataset_address/Address_Datasets/dataset3.csv')
ds4 = pd.read_csv('dataset_address/Address_Datasets/dataset4.csv')
ds5 = pd.read_csv('dataset_address/Address_Datasets/dataset5.csv')
ds6 = pd.read_csv('dataset_address/Address_Datasets/dataset6.csv')
ds7 = pd.read_csv('dataset_address/Address_Datasets/dataset7.csv')
ds8 = pd.read_csv('dataset_address/Address_Datasets/dataset8.csv')
ds9 = pd.read_csv('dataset_address/Address_Datasets/dataset9.csv')
ds10 = pd.read_csv('dataset_address/Address_Datasets/dataset10.csv')
ct911 = pd.read_csv('dataset_address/Address_Datasets/CT911_feb2026.csv')
parcel_CAMA = pd.read_csv('dataset_address/Address_Datasets/Parcel_CAMA_2025.csv')

datasets = [ds1, ds2, ds3, ds4, ds5, ds6, ds7, ds8, ds9, ds10, ct911, parcel_CAMA]


---

## Stage 1 — Problem Framing & Target Definition  *(5 pts)*

**Produce in writing (no code required):**
- One-paragraph problem statement: what would a model predict, for whom, and why?
- Explicit target definition: variable name, type (binary / multiclass / continuous / count / ordinal), unit of observation, **time-of-prediction** (what is known *at prediction time* vs. observed afterward).
- Proposed primary evaluation metric **with justification** (why F1 vs. ROC-AUC vs. PR-AUC; why RMSE vs. MAE vs. MAPE) based on what you know of the problem at this early stage.
- Statement of single- vs. multi-target / multi-output problem based on what you know of the problem at this early stage.

### Your Stage 1 response

*Write your problem framing here.*

**Problem statement:**

The Main goal of this model is to validate the different addresses so that emergency services among the 911 calls can know for sure that the address they are going to actually exists. This model will aim to confirm the existence of addresses and raise the others as needing review to confirm that they still exist. This would help dispatchers sending the emergency services to the right location should they time arise. Additionally, the GIS department of the Connecticut government will benefit from this model as they are the ones in contact with 911 services and create the master list of all addresses in the state of Connecticut.


**Target definition:**

*Datasets 1-5*
- ADDR_STT_NUM: Street Number
- STTPFX_DRTN_CD: Street Prefix Direction
- ADDR_STT_NM: Street Name
- STT_TYP_CD: Street code/ type (Rd, Ln, Dr, etc.)
- STTSFX_DRTN_CD: Street Suffix Direction Code ("NW", "SE", etc.)
- ADDR_UNIT_TYP_CD: Unit Type code (APT, UNIT, STE)
- ADDR_UNIT_VAL_TXT: Unit Number or Name
- MNCPLT_NM: Name of Municipality
- ST_CD: State Code (CT)
- ZIP_CD_NUM: Zip Code
- SLONG: Longitude
- SLAT: Latitude

*Dataset 6*
- DATECREATE: Date the record was created in the database or GIS
- DATEMODIFI: Date the record was last modified
- INSTALLATI: Date of installation or ID referencing install event
- LOCATIONID: Unique ID for the physical service location
- GISONUMBER: GIS-assigned number
- TOWN: Name of town or Municipality
- CUSTOMERNA: additional details about location
- PERMISEHIG: high end of premise address range
- PREMISELOW: low end of premise address range
- LIFESUPPOR: flag indicating if life-support is used at the address
- SERVICE_AD: Full service address string
- CITY_NM: City Name
- PROV_STATE: State
- POSTAL_CD: Zip code
- REFRESH_DT: Date the record was last refreshed

*Dataset 7*
- AddressLine1: Primary address line
- AddressLine2: Secondary address line
- HouseAddress: Parsed house number portion of the address
- Apt: Apartment or unit number
- Street: street name or type
- City: City or locality name
- St: state abbreviation (CT)
- Zip: ZIP code
- CustTypeDescription: Description of customer type (Residential vs Commercial)

*Dataset 8*
- StreetAdd: Full Street Address
- State: State
- Zip: Zipcode

*Dataset 9*
- Full address
- Latitude
- Longitude

*Dataset 10*
- ID: Unique identifier for the address record
- Num: House or street number
- UNIT: Unit, Appartment, or suite Designation
- ST: Street Name
- NGHB: Neighborhood name
- MUNI: Municipality name (town or city)
- STATE: State (CT)
- ZIP5: 5-digit Zip code
- ZIP4: 4-digit Zip+4 extension


**Time-of-prediction:** 

We can know if an address is valid or not by looking it up on a map. However, this is a very tedious process and there just isn't enough time in the day to perform such a task. We know that Dataset 10 is a validated dataset and we can use that dataset, along with current CT911 and Parcel CAMA to match the addresses in order to confirm if they are valid or not and flag the others for review shoudl they need a closer look.

**Proposed metric and justification:** 

The primary form of evaluation for this model will be a classification of how likely each address is to be valid. from there we can look at the different sources that the addresss is in, what is missing from each individual address, and whether or not the inputs in the datasets make sense to have an address at that location (any specific missing/invalid entries). Once all of the addresses have been classified as needing review or probably valid, a comparison between those that are classified under that title will reveal how well the model performed.

**Single- vs. multi-target:**

From my understanding, we are not looking at a specific variable output. We are looking for a classification of each addresses and a confidence of how sure we are that an address is valid. The classification can be as simple as needs review / likely valid but having something such black and white doesn't make sense. It would be best if there were multiple categories such as likely invalid, needing review, and likely valid. The needing review category would be a sign the someone needs to go in and physically review the address to confirm its existance or confirm its removal.


---

## Stage 2 — Data Inventory & Schema Audit  *(7 pts)*

**Produce:**
- Shape, memory footprint, dtype table.
- A **semantic feature-type table** (one row per column): `name | pandas dtype | semantic type | role | notes`, where  
  `semantic type ∈ {continuous, count, ordinal, nominal, binary, datetime, text, identifier, geographic, other}`  
  `role ∈ {feature, target, identifier, metadata, leakage-suspect}`.
- A list of columns where pandas dtype disagrees with the semantic type (e.g., zip codes as `int64`) and your planned coercion.

This table seeds your final Data Card.

In [ ]:
# Shape, memory, dtypes

for i in range(len(datasets)):
    dataset = datasets[i]
    if i == 10:
        print(f'Dataset: CT911')
    elif i == 11:
        print(f'Dataset: parcel_CAMA')
    else:
        print(f'Dataset: {i+1}')
    print(f'Shape: {dataset.shape}')
    print(f'Memory: {dataset.memory_usage().sum()}')
    dtypes_table = (
        dataset.dtypes
        .reset_index()
        .rename(columns = {
            "index": "Column Name",
            0: "pandas_dtype"
        })
    )
    print(dtypes_table)
    print('\n')




In [ ]:
# Semantic feature-type table (one row per column)
SEMANTIC_TYPES = {
    "continuous",
    "count",
    "ordinal",
    "nominal",
    "binary",
    "datetime",
    "text",
    "identifier",
    "geographic",
    "other",
}

ROLES = {
    "feature",
    "target",
    "identifier",
    "metadata",
    "leakage-suspect",
}

def infer_semantic_type(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_string_dtype(series):
        nunique = series.nunique(dropna=True)

        if nunique > 0.9 * len(series):
            return "identifier"

        if nunique == 2:
            return "binary"

        avg_len = series.dropna().astype(str).str.len().mean()
        if avg_len and avg_len > 30:
            return "text"

        return "nominal"

    if pd.api.types.is_bool_dtype(series):
        return "binary"

    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)

        if nunique == 2:
            return "binary"

        if pd.api.types.is_integer_dtype(series):
            if series.min() >= 0:
                return "count"
            return "ordinal"

        return "continuous"

    return "other"


def infer_role(column_name, target=None):
    col = column_name.lower()

    if target and column_name == target:
        return "target"

    if col.endswith("_id") or col == "id":
        return "identifier"

    if any(x in col for x in ["created", "updated", "timestamp"]):
        return "metadata"

    return "feature"


def create_feature_table(df, target=None):
    rows = []

    for col in df.columns:
        rows.append({
            "name": col,
            "pandas dtype": str(df[col].dtype),
            "semantic type": infer_semantic_type(df[col]),
            "role": infer_role(col, target),
            "notes": ""
        })

    return pd.DataFrame(rows)


for i in range(len(datasets)):
    dataset = datasets[i]
    if i == 10:
        print("Dataset: CT911")
    elif i == 11:
        print("Dataset: parcel_CAMA")
    else:
        print(f"Dataset: {i+1}")
    
    feature_table  = create_feature_table(dataset)
    print(feature_table)
    print('\n')



In [ ]:
# Dtype-vs-semantic disagreements + coercion plan
def find_dtype_semantic_mismatches(semantic_table):
    issues = []

    for _, row in semantic_table.iterrows():

        col = row["name"]
        dtype = str(row["pandas dtype"])
        semantic = row["semantic type"]

        issue = None
        coercion = None

        if semantic == "datetime" and not dtype.startswith("datetime"):
            issue = "Date/time stored as non-datetime"
            coercion = f"df['{col}'] = pd.to_datetime(df['{col}'])"

        elif semantic == "identifier" and dtype != "object":
            issue = "Identifier should not be numeric"
            coercion = f"df['{col}'] = df['{col}'].astype(str)"

        elif semantic == "geographic" and dtype != "object":
            issue = "Geographic code stored numerically"
            coercion = f"df['{col}'] = df['{col}'].astype(str)"

        elif semantic == "binary" and dtype not in ["bool", "category"]:
            issue = "Binary variable stored as numeric/text"
            coercion = (
                f"df['{col}'] = df['{col}'].astype('category')"
            )

        elif semantic in ["nominal", "ordinal"] and dtype != "category":
            issue = "Categorical variable not encoded as category"
            coercion = (
                f"df['{col}'] = df['{col}'].astype('category')"
            )

        if issue:
            issues.append({
                "column": col,
                "dtype": dtype,
                "semantic_type": semantic,
                "issue": issue,
                "planned_coercion": coercion
            })

    return pd.DataFrame(issues)

for i in range(len(datasets)):
    dataset = datasets[i]
    if i == 10:
        print("Dataset: CT911")
    elif i == 11:
        print("Dataset: parcel_CAMA")
    else:
        print(f"Dataset: {i+1}")

    feature_table = create_feature_table(dataset)
    mismatches = find_dtype_semantic_mismatches(feature_table)
    print(mismatches)
    print('\n')


### Stage 2 — Interpretation

*Summarize what the schema tells you about this dataset, and list the coercions you will apply downstream.*

*Most of the datasets are filled with non-numeric data. Address Number and zipcodes are the only real numeric variables throughout the datasets. While some of the datasets have numeric data, such as land acres in the Parcel_CAMA dataset, the two fields that are consistent throughout all 12 datasets are the street number and zipcodes. Of the string variables, municipality and street name are among the more common fields. Some of the datasets provide the longitude and latitude for the parcel to confirm the geographical coordinates of the address.*

---

## Stage 3 — Data Quality Assessment  *(8 pts)*

**Produce:**
- Exact and approximate (near-)duplicate-row counts; justify your approximate-duplicate key.
- Constant / quasi-constant columns (variance ≈ 0 or single value > 95%).
- ID-like columns (cardinality ≈ row count).
- Impossible / out-of-domain values (negatives where impossible, future dates, percentages > 100, etc.).
- Encoding glitches: whitespace, mixed case, mojibake, inconsistent units.
- For each issue: severity (low / medium / high) and remediation (drop / cap / impute / flag).

In [ ]:
# Duplicates (exact + approximate), constant cols, ID-like cols
exact_duplicates = []
for i in range(len(datasets)):
    exact_duplicates.append(datasets[i].duplicated().sum())
print(exact_duplicates)

fig, ax = plt.subplots()
ax.bar(range(1, len(exact_duplicates) + 1), exact_duplicates)
ax.set_xlabel("Dataset")
ax.set_ylabel("Number of Exact Duplicates")
ax.set_title("Exact Duplicates by Dataset")
plt.show()

# Approximate duplicates
def duplicate_records(df ,dataset_name):
    n_rows = len(df)
    exact_dup_rows = df.duplicated().sum()
    candidate_keys = []
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        uniqueness = n_unique / n_rows if n_rows else 0
        dup_count = df[col].duplicated().sum()
        candidate_keys.append({
            "column": col,
            "unique_values": n_unique,
            "uniqueness_ratio": uniqueness,
            "duplicate_count": dup_count
        })
    candidate_keys = (
        pd.DataFrame(candidate_keys)
        .sort_values("uniqueness_ratio", ascending=False)
    )
    best_key = candidate_keys.iloc[0]
    return {
        "dataset": dataset_name,
        "rows": n_rows,
        "exact_duplicate_rows": exact_dup_rows,
        "best_approximate_key": best_key["column"],
        "key_uniqueness_ratio": round(best_key["uniqueness_ratio"], 4),
        "key_duplicate_count": int(best_key["duplicate_count"])
    }


In [ ]:
# Constant/Quasi constant columns
def detect_constant_columns(df, col):
    unique_values = df[col].nunique()
    total_rows = len(df)
    if unique_values == 1:
        return {"is_constant": True, "unique_values": unique_values}
    elif unique_values == 2:
        return {"is_quasi_constant": True, "unique_values": unique_values}
    else:
        return {"is_constant": False, "is_quasi_constant": False, "unique_values": unique_values}

for i in range(len(datasets)):
    for col in datasets[i].columns:
        if pd.api.types.is_string_dtype(datasets[i][col]):
            const_issues = detect_constant_columns(datasets[i], col)
            if const_issues.get("is_constant", False) or const_issues.get("is_quasi_constant", False):
                print(f"Dataset: {i+1}, Column: {col}")
                print(const_issues)
                print()

In [ ]:
# Find the best composite key for a dataset
results = []

for i, df in enumerate(datasets):
    results.append(
        duplicate_records(df, f"dataset_{i+1}")
    )

summary = pd.DataFrame(results)

print(summary)

def find_approximate_key(df,
                         target_uniqueness=0.99,
                         max_columns=5):

    n = len(df)

    scores = {
        col: df[col].nunique(dropna=False) / n
        for col in df.columns
    }

    key = [max(scores, key=scores.get)]
    current_uniqueness = scores[key[0]]

    remaining = set(df.columns) - set(key)

    while (
        current_uniqueness < target_uniqueness
        and len(key) < max_columns
        and remaining
    ):

        best_col = None
        best_uniqueness = current_uniqueness

        for col in remaining:

            candidate = key + [col]

            uniqueness = (
                df[candidate]
                .drop_duplicates()
                .shape[0]
                / n
            )

            if uniqueness > best_uniqueness:
                best_uniqueness = uniqueness
                best_col = col

        if best_col is None:
            break

        key.append(best_col)
        remaining.remove(best_col)
        current_uniqueness = best_uniqueness

    return {
        "key_columns": key,
        "uniqueness_ratio": current_uniqueness,
        "duplicate_count": int(
            n - current_uniqueness * n
        )
    }

for df in datasets:
    result = find_approximate_key(df)
    print(result)

In [ ]:
# Id-like columns (Cardinality ~ row count)
def find_id_candidates(df, threshold=0.99):

    n_rows = len(df)

    results = []

    for col in df.columns:

        cardinality = df[col].nunique(dropna=True)

        uniqueness_ratio = cardinality / n_rows

        if uniqueness_ratio >= threshold:

            results.append({
                "column": col,
                "cardinality": cardinality,
                "row_count": n_rows,
                "uniqueness_ratio": round(
                    uniqueness_ratio, 4
                )
            })

    return pd.DataFrame(results)

for df in datasets:
    id_candidates = find_id_candidates(df)
    print(id_candidates)
    print('\n')

In [ ]:
# Impossible / out-of-domain values
GEOGRAPHIC_NEGATIVE_ALLOWED = {
    "LONG", "LAT","SLONG", "SLAT","LONGITUDE", "LATITUDE", "X_LONG", "Y_LAT"
}

def find_domain_violations(df):
    violations = []
    today = pd.Timestamp.today()

    for col in df.columns:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):
            if col.upper() in GEOGRAPHIC_NEGATIVE_ALLOWED:
                pass
            else:
                negative_count = (s < 0).sum()
                if negative_count:
                    violations.append({
                        "column": col,
                        "rule": "negative values",
                        "count": int(negative_count),
                        "examples": s[s < 0].head(5).tolist()
                    })

                if any(
                    token in col.lower()
                    for token in ["pct", "percent", "percentage", "rate"]
                ):
                    invalid_pct = ((s < 0) | (s > 100)).sum()
                    if invalid_pct:
                        violations.append({
                            "column": col,
                            "rule": "percentage outside [0,100]",
                            "count": int(invalid_pct),
                            "examples": s[(s < 0) | (s > 100)].head(5).tolist()
                        })

        if pd.api.types.is_datetime64_any_dtype(s):
            future_count = (s > today).sum()
            if future_count:
                violations.append({
                    "column": col,
                    "rule": "future date",
                    "count": int(future_count),
                    "examples": s[s > today].head(5).tolist()
                })

    return pd.DataFrame(violations)

for i in range(len(datasets)):
    violations = find_domain_violations(datasets[i])
    print(f"Dataset: {i+1}")
    print(violations)
    print('\n')

In [ ]:
# Encoding glitches (whitespace, case, mojibake, units)
# White space issues
def detect_whitespace_issues(df, col):
    s = df[col].astype(str)

    leading = s.str.match(r"^\s+")
    trailing = s.str.match(r".*\s+$")
    internal = s.str.contains(r"\s{2,}")  # multiple spaces

    return {
        "leading": leading.sum(),
        "trailing": trailing.sum(),
        "internal_multiple_spaces": internal.sum()
    }

for i in range(len(datasets)):
    for col in datasets[i].columns:
        if pd.api.types.is_string_dtype(datasets[i][col]):
            whitespace_issues = detect_whitespace_issues(datasets[i], col)
            if any(whitespace_issues.values()):
                print(f"Dataset: {i+1}, Column: {col}")
                print(whitespace_issues)
                print()

In [ ]:
# Inconsistent Casing
def detect_case_issues(df, col):

    s = df[col].dropna().astype(str)

    n_upper = s.str.isupper().sum()
    n_lower = s.str.islower().sum()
    n_title = s.str.istitle().sum()

    total = len(s)

    return {
        "mixed_case_ratio": 1 - max(n_upper, n_lower, n_title) / total
    }

for i in range(len(datasets)):
    for col in datasets[i].columns:
        if pd.api.types.is_string_dtype(datasets[i][col]):
            case_issues = detect_case_issues(datasets[i], col)
            if case_issues["mixed_case_ratio"] > 0:
                print(f"Dataset: {i+1}, Column: {col}")
                print(case_issues)
                print()

In [ ]:
# Mojibake issues
def detect_mojibake(df, col):
    s = df[col].astype(str)
    patterns = [
        r"Ã.|â€™|â€œ|â€�",   # classic UTF-8 corruption
        r"[^\x00-\x7F]{2,}"  # unusual multi-byte sequences
    ]

    mask = False
    for p in patterns:
        mask = mask | s.str.contains(p, regex=True)

    return {
        "mojibake_count": mask.sum(),
        "examples": df.loc[mask, col].head(5).tolist()
    }

for i in range(len(datasets)):
    for col in datasets[i].columns:
        if pd.api.types.is_string_dtype(datasets[i][col]):
            mojibake_issues = detect_mojibake(datasets[i], col)
            if mojibake_issues["mojibake_count"] > 0:
                print(f"Dataset: {i+1}, Column: {col}")
                print(mojibake_issues)
                print()

In [ ]:
#Inconsistent units
def detect_unit_inconsistency(df, col):
    s = df[col].astype(str)
    units = s.str.extract(r"([a-zA-Z]+)$", expand=False).str.lower()
    unit_counts = units.value_counts(dropna=True)
    return {
        "unique_units": unit_counts.to_dict(),
        "num_units": unit_counts.shape[0]
    }

for i in range(len(datasets)):
    for col in datasets[i].columns:
        if pd.api.types.is_string_dtype(datasets[i][col]):
            unit_issues = detect_unit_inconsistency(datasets[i], col)
            if unit_issues["num_units"] > 1:
                print(f"Dataset: {i+1}, Column: {col}")
                print(unit_issues)
                print()

In [ ]:
#Severity levels and remediations
def assess_exact_duplicates(df):
    rate = df.duplicated().mean()

    if rate < 0.01:
        severity = "low"
    elif rate < 0.05:
        severity = "medium"
    else:
        severity = "high"

    return {
        "issue": "exact_duplicates",
        "rate": rate,
        "severity": severity,
        "remediation": "flag"
    }

def assess_approx_duplicates(uniqueness):
    if uniqueness > 0.99:
        severity = "low"
    elif uniqueness > 0.95:
        severity = "medium"
    else:
        severity = "high"

    return {
        "issue": "approx_duplicates",
        "severity": severity,
        "remediation": "flag"
    }

def assess_constant(col, df):
    n_unique = df[col].nunique(dropna=True)
    ratio = n_unique / len(df)

    if n_unique == 1:
        severity = "high"
    elif ratio < 0.01:
        severity = "medium"
    elif ratio < 0.05:
        severity = "low"
    else:
        return None

    return {
        "issue": "constant_or_quasi_constant",
        "severity": severity,
        "remediation": "flag"
    }

def assess_id_like(col, df):
    ratio = df[col].nunique(dropna=True) / len(df)

    if ratio > 0.99:

        severity = "medium"

        return {
            "issue": "id_like_column",
            "severity": severity,
            "remediation": "flag"
        }

    return None

def assess_impossible_values(mask, issue_name):
    rate = mask.mean()

    if rate < 0.01:
        severity = "low"
    elif rate < 0.05:
        severity = "medium"
    else:
        severity = "high"

    return {
        "issue": issue_name,
        "severity": severity,
        "remediation": "flag"
    }


def assess_encoding_issues(mask, issue_name):
    rate = mask.mean()

    if rate < 0.05:
        severity = "low"
    elif rate < 0.2:
        severity = "medium"
    else:
        severity = "high"

    return {
        "issue": issue_name,
        "severity": severity,
        "remediation": "flag"
    }

def build_data_quality_report(df):
    issues = []

    dup_rate = df.duplicated().mean()
    issues.append({
        "issue": "exact_duplicates",
        "severity": "high" if dup_rate > 0.05 else "medium" if dup_rate > 0.01 else "low",
        "remediation": "flag"
    })

    for col in df.columns:

        s = df[col]
        n = len(df)
        missing_rate = s.isna().mean()
        if missing_rate == 1.0:
            issues.append({
                "column": col,
                "issue": "fully_missing_column",
                "severity": "high",
                "remediation": "drop"
            })
            continue
        nunique = s.nunique(dropna=True) / len(df)
        ratio = nunique / n if n else 0

        if ratio < 0.05:
            issues.append({
                "column": col,
                "issue": "quasi_constant",
                "severity": "medium" if ratio < 0.01 else "low",
                "remediation": "flag"
            })
        
        if ratio > 0.99:
            issues.append({
                "column": col,
                "issue": "id_like",
                "severity": "medium",
                "remediation": "flag"
            })

        if df[col].dtype == "object":
            ws_mask = df[col].astype(str).str.contains(r"^\s|\s$", regex=True)
            rate = ws_mask.mean()

            if rate > 0:
                issues.append({
                    "column": col,
                    "issue": "whitespace",
                    "severity": "low" if rate < 0.05 else "medium",
                    "remediation": "flag"
                })

    return pd.DataFrame(issues)

for i in range(len(datasets)):
    print(f"Dataset: {i+1}")
    print(build_data_quality_report(datasets[i]))
    print()


### Stage 3 — Issue Register

| Issue | Affected column(s) | Severity | Remediation |
|---|---|---|---|
|  |  |  |  |

Some of the categories, like street type, have have somewhat similar data in that there are only a few street types that streets are named after, such as street, drive, road etc. While these are good to flag as there could be other issues within the data. All state codes are the same as we are only looking at the state of CT so there is no need to worry that all entries say Connecticut. There are a few datasets that have duplicate records and that is a definitive cause for review of them. 


---

## Stage 4 — Missingness Analysis  *(8 pts)*

**Produce:**
- Per-column missing count and percentage; row-completeness distribution.
- Missingness matrix and missingness-correlation heatmap (e.g., `missingno`).
- **Missingness-vs-target test** for each column with non-trivial missingness:
  - Classification target: chi-square (with Cramér's V).
  - Regression target: Mann–Whitney or Kruskal–Wallis (with effect size, e.g., η² or rank-biserial).
- A reasoned MCAR / MAR / MNAR posture for the most-missing columns (argue, don't assert).
- Per-column imputation plan with justification.

In [ ]:
# Missingness counts, percentages, row-completeness distribution
def dataset_quality_metrics(df, name="dataset"):
    n_rows = len(df)
    n_cols = df.shape[1]

    missing_count = df.isna().sum()
    missing_pct = (missing_count / n_rows) * 100

    col_summary = pd.DataFrame({
        "dataset": name,
        "column": df.columns,
        "missing_count": missing_count.values,
        "missing_pct": missing_pct.values
    })

    row_non_null_count = df.notna().sum(axis=1)
    row_completeness = (row_non_null_count / n_cols) * 100

    completeness_distribution = (
        row_completeness.round(0)
        .value_counts()
        .sort_index()
        .reset_index()
    )
    completeness_distribution.columns = ["row_completeness_pct", "row_count"]
    completeness_distribution["dataset"] = name

    return col_summary, completeness_distribution

for i in range(len(datasets)):
    dataset = datasets[i]
    if i == 10:
        print("Dataset: CT911")
    elif i == 11:
        print("Dataset: parcel_CAMA")
    else:
        print(f"Dataset: {i+1}")
    col_summary, completeness_distribution = dataset_quality_metrics(datasets[i], name=f"dataset_{i+1}")
    print(col_summary)
    print(completeness_distribution)
    print()

In [ ]:
# Graph showing for dataset 1
plt.figure(figsize=(10, 6))
plt.title("Dataset 1")
plt.xlabel("Columns")
plt.ylabel("Missing Values")
plt.bar(range(len(datasets[0].columns)), datasets[0].isnull().mean()*100)
plt.xticks(range(len(datasets[0].columns)), datasets[0].columns, rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Missingness matrix
def missingness_matrix(df):
    return df.notna().astype(int)

for i in range(len(datasets)):
    print(missingness_matrix(datasets[i]))

In [ ]:
# Correlation heatmap
def plot_missingness_heatmap(df, title="Missingness Correlation Heatmap"):
    plt.figure(figsize=(10, 6))
    msno.heatmap(df)
    plt.title(title)
    plt.show()

for i in range(len(datasets)):
    plot_missingness_heatmap(datasets[i], title=f"Missingness Correlation Heatmap - Dataset {i+1}")

In [ ]:
# Missingness-vs-target tests with effect sizes
def get_missing_cols(df, threshold=0.01):
    return df.columns[df.isna().mean() > threshold].tolist()

def cramers_v_test(x,y):

    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan

    try:
        chi2, p, dof, expected = chi2_contingency(table)
    except ValueError:
        return np.nan

    n = table.to_numpy().sum()
    if n == 0:
        return np.nan
    
    r, k = table.shape
    cramers_v = np.sqrt(
        chi2 / (n * min(r-1, k-1))
    )

    return cramers_v

def missingness_cramers_v_matrix(df):
    missing_df = df.isna().astype(int)
    cols = missing_df.columns
    result = pd.DataFrame(
        np.nan,
        index = cols,
        columns = cols
    )
    for col1 in cols:
        for col2 in cols:
            if col1 == col2:
                result.loc[col1, col2] = 1.0
            else:
                result.loc[col1, col2] = cramers_v_test(
                    missing_df[col1], 
                    missing_df[col2]
                )
    return result

for i in range(len(datasets)):
    missing_cols = get_missing_cols(datasets[i])
    print(f"Columns missing data in dataset {i+1}:")
    print(missing_cols)
    print()

In [ ]:
for i in range(len(datasets)):
    v_matrix = missingness_cramers_v_matrix(datasets[i])
    if i == 10:
        print(f'V-matrix for CT911 Dataset:')
    elif i == 11:
        print(f'V-matrix for Parcel_CAMA Dataset:')
    else:
        print(f'V-matrix for Dataset {i+1}:')
    print(v_matrix)
    print()

    missing_pct = datasets[i].isna().mean().sort_values(ascending=False)
    most_missing = (
        missing_pct[missing_pct > .10]
        .index
    )
    print(f"Most missing columns in dataset {i+1}:")
    print(most_missing)
    print()

In [ ]:
# MCAR / MAR / MNAR

def build_missingness_report(df, v_matrix, mar_threshold=0.30, weak_mar_threshold=0.10):
    missing_pct = df.isna().mean()

    report_rows = []

    for col in df.columns:

        pct = missing_pct[col]

        # --- handle 100% missing ---
        if pct == 1.0:
            report_rows.append({
                "column": col,
                "missing_pct": pct,
                "mechanism": "Not assessable",
                "reasoning": "Column is entirely missing; no statistical structure can be evaluated.",
                "imputation_strategy": "Drop",
                "justification": "No observed values exist to support imputation."
            })
            continue

        # --- compute max missingness association ---
        if col in v_matrix.columns:
            max_v = v_matrix[col].drop(col, errors="ignore").max()
        else:
            max_v = np.nan

        # --- mechanism logic ---
        if np.isnan(max_v):
            mechanism = "Unclear"
            reasoning = "No valid missingness associations could be computed."
            strategy = "Manual review"
            justification = "Insufficient information for automated classification."

        elif pct > 0.5 and max_v < weak_mar_threshold:
            mechanism = "MNAR-consistent"
            reasoning = (
                f"High missingness ({pct:.2%}) with weak associations "
                f"(max Cramér's V={max_v:.2f}) suggests missingness may depend "
                "on unobserved values."
            )
            strategy = "Missingness flag + model-based imputation"
            justification = "High missingness increases risk that missingness is value-dependent."

        elif max_v >= mar_threshold:
            mechanism = "MAR (suggested)"
            reasoning = (
                f"Moderate to strong associations between missingness and other variables "
                f"(max Cramér's V={max_v:.2f}) suggest dependence on observed data."
            )
            strategy = "MICE / model-based imputation"
            justification = "Observed variables likely informative for predicting missing values."

        elif max_v >= weak_mar_threshold:
            mechanism = "Weak MAR evidence"
            reasoning = (
                f"Weak associations detected (max Cramér's V={max_v:.2f}). "
                "Some dependence on observed variables may exist."
            )
            strategy = "Simple + model-based hybrid imputation"
            justification = "Unclear strength of missingness dependency."

        else:
            mechanism = "MCAR-consistent"
            reasoning = (
                f"No meaningful association between missingness and other variables "
                f"(max Cramér's V={max_v:.2f})."
            )
            strategy = "Mean/median (numeric) or mode (categorical)"
            justification = "Missingness appears approximately random in observed data."

        report_rows.append({
            "column": col,
            "missing_pct": pct,
            "mechanism": mechanism,
            "reasoning": reasoning,
            "imputation_strategy": strategy,
            "justification": justification
        })

    return pd.DataFrame(report_rows).sort_values("missing_pct", ascending=False)

for i in range(len(datasets)):
    df = datasets[i]
    v_matrix = missingness_cramers_v_matrix(df)
    report = build_missingness_report(df, v_matrix)
    if i == 10:
        print(f'Report for CT911 Dataset:')
    elif i == 11:
        print(f'Report for Parcel_CAMA Dataset:')
    else:
        print(f'Report for Dataset {i+1}:')
    display(report)

### Stage 4 — Imputation Plan

| Column | % missing | Mechanism (MCAR/MAR/MNAR) + reasoning | Imputation strategy | Justification |
|---|---|---|---|---|
|  |  |  |  |  |


There are several columns in each dataset that are missing a significant portion of their data. When looking at the missingness of each of the variables, it looks as though "STTSFX_DRTN_CD" and "STTPFX_DRTN_CD" have the highest percentage of missingness among cvariables across all 12 datasets. This should be considered when looking into what should and shouldn't have an effect on the validity of each of the addresses. If an address has a missing value for "STTPFX_DRTN_CD", then it shouldn't be weighed as heavily than if it had a missing value for address number, for example.

---

## Stage 5 — Univariate Analysis  *(8 pts)*

**Numeric features:**
- Summary statistics including count, mean, median, std, min, max, IQR, **skewness**, **kurtosis**.
- Distribution plots (histogram + KDE, or ECDF) for at least 8 numeric features.
- For features with |skew| > 1 or heavy tails: propose a transformation (log, log1p, Box-Cox, Yeo-Johnson) and **show the post-transformation distribution**.

**Categorical features:**
- Cardinality table.
- Frequency tables for low-cardinality categoricals.
- **Rare-level audit** for high-cardinality categoricals: define your "rare" threshold and propose grouping into "Other".
- Identify any categoricals masquerading as numeric (e.g., zip codes).

In [ ]:
# Numeric: summary stats including skew & kurtosis
def keep_only_numeric_like_columns(df):
    numeric_df = pd.DataFrame(index=df.index)
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        original_non_null = df[col].notna().sum()
        numeric_ratio = converted.notna().sum() / original_non_null if original_non_null else 0

        if numeric_ratio > 0.95:
            numeric_df[col] = converted
        else:
            continue

    # dropping constant and quasi-constant columns
    for col in numeric_df.columns:
        if numeric_df[col].unique().size == 1:
            numeric_df.drop(columns=[col], inplace=True)
        elif numeric_df[col].unique().size == 2:
            numeric_df.drop(columns=[col], inplace=True)
    
    return numeric_df


def summary_stats(df):
    results = []
    for column in df.columns:
        s = df[column].dropna()
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
    
        results.append({
            "column": column,
            "count": s.count(),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "max": s.max(),
            "IQR": q3 - q1,
            "skew": skew(s, bias = False) if len(s) > 2 else np.nan,
            "kurtosis": kurtosis(s, fisher=True, bias=False) if len(s) > 3 else np.nan,
        })

    return pd.DataFrame(results).round(4)

for i in range(len(datasets)):
    df = datasets[i]
    numeric_df = keep_only_numeric_like_columns(df)
    summary = summary_stats(numeric_df)
    if i == 10:
        print(f'Summary stats for CT911 Dataset:')
    elif i == 11:
        print(f'Summary stats for Parcel_CAMA Dataset:')
    else:
        print(f'Summary stats for Dataset {i+1}:')
    print(summary)
    print()

In [ ]:
# Numeric: distribution plots (8+ features)
def plot_distributions(df, n_features=8):
    cols_to_plot = df.columns
    n = len(cols_to_plot)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(12, 4 * nrows)
    )

    axes = np.array(axes).flatten()

    for ax, col in zip(axes, cols_to_plot):
        sns.histplot(
            df[col].dropna(),
            kde=True,
            ax=ax
        )

        ax.set_title(f"Distribution of {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")

    # Remove unused axes
    for ax in axes[len(cols_to_plot):]:
        fig.delaxes(ax)

    plt.tight_layout()
    plt.show()

for i in range(len(datasets)):
    numeric_df = keep_only_numeric_like_columns(datasets[i])
    if i < 9:
        plot_distributions(numeric_df)
    


In [ ]:
# Numeric: transformation candidates (before/after)
def transform_and_plot_skewed_features(df, skew_df, col_col = "column",skew_col = "skew", skew_threshold = 1.0):

    df_transformed = df.copy()
    results = {}

    for _, row in skew_df.iterrows():
        col = row[col_col]
        skew = row[skew_col]

        if col not in df.columns:
            continue

        if np.isnan(skew):
            continue

        if abs(skew) <= skew_threshold:
            continue

        x = df[col].dropna()

        if x.nunique() < 2:
            continue

        transformed = None
        method = None

        if (x > 0).all():
            try:
                transformed, lam = stats.boxcox(x)
                method = f"Box-Cox (lambda={lam:.3f})"
            except Exception:
                transformed = np.log1p(x)
                method = "log1p"

        elif (x >= 0).all():
            transformed = np.log1p(x)
            method = "log1p"

        else:
            pt = PowerTransformer(method="yeo-johnson", standardize=False)
            transformed = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
            method = "Yeo-Johnson"

        df_transformed.loc[x.index, col] = transformed

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        axes[0].hist(x, bins=30, edgecolor="black")
        axes[0].set_title(f"Before: {col}\nskew={skew:.2f}")

        axes[1].hist(transformed, bins=30, edgecolor="black")
        axes[1].set_title(f"After: {col}\n{method}")

        plt.tight_layout()
        plt.show()

        results[col] = {
            "original_skew": skew,
            "method": method
        }

    return df_transformed, pd.DataFrame.from_dict(results, orient="index")

for i in range(len(datasets)):
    df = datasets[i]
    numeric_df = keep_only_numeric_like_columns(df)
    summary = summary_stats(numeric_df)
    if i == 10:
        print(f'Transformations for CT911 dataset')
    elif i == 11:
        print(f'Transformations for Parcel_CAMA dataset')
    else:    
        print(f'Transformations for dataset {i+1}')
    transform_and_plot_skewed_features(numeric_df, summary)

In [ ]:
# Categorical: cardinality
def create_cardinality_table(df, high_cardinality_threshold = 0.5):
    n_rows = len(df)

    summary = []
    for col in df.columns:
        col_data = df[col]

        n_unique = col_data.nunique(dropna=True)
        n_missing = col_data.isna().sum()
        dtype = col_data.dtype

        cardinality_ratio = n_unique / n_rows if n_rows > 0 else 0

        if cardinality_ratio >= high_cardinality_threshold:
            card_flag = "High"
        elif cardinality_ratio >= 0.1:
            card_flag = "Medium"
        else:
            card_flag = "Low"

        summary.append({
            "column": col,
            "dtype": str(dtype),
            "missing_count": n_missing,
            "missing_pct": round(n_missing / n_rows * 100, 2) if n_rows > 0 else 0,
            "unique_values": n_unique,
            "cardinality_ratio": round(cardinality_ratio, 4),
            "cardinality_level": card_flag
        })

    return pd.DataFrame(summary).sort_values(
        by="cardinality_ratio",
        ascending=False
    ).reset_index(drop=True)

for i in range(len(datasets)):
    cardinality_table = create_cardinality_table(datasets[i])
    if i == 10:
        print(f'Cardinality table for CT911 dataset')
    elif i == 11:
        print(f'Cardinality table for Parcel_CAMA dataset')
    else:    
        print(f'Cardinality table for dataset {i+1}:')
    display(cardinality_table)

In [ ]:
# Frequency tables
def low_cardinality_frequency_from_table(cardinality_df,original_df,column_col = "column",level_col = "cardinality_level",target_level = "Low",normalize = True,top_n = None):


    freq_tables = {}

    low_cols = cardinality_df.loc[
        cardinality_df[level_col] == target_level,
        column_col
    ]

    for col in low_cols:
        if col not in original_df.columns:
            continue

        freq = original_df[col].value_counts(dropna=False)

        if normalize:
            freq = pd.concat([
                freq.rename("count"),
                (freq / len(original_df)).rename("percent")
            ], axis=1)
        else:
            freq = freq.to_frame("count")

        freq.index.name = col
        freq = freq.reset_index()

        if top_n is not None:
            freq = freq.head(top_n)

        freq_tables[col] = freq

    return freq_tables

for i in range(len(datasets)):
    cardinality_df = create_cardinality_table(datasets[i])
    freq_df = low_cardinality_frequency_from_table(cardinality_df, datasets[i])
    if i == 10:
        print(f'Frequency table for CT911 dataset')
    elif i == 11:
        print(f'Frequency table for Parcel_CAMA dataset')
    else:    
        print(f'Frequency table for dataset {i+1}:')
    display(freq_df)


In [ ]:
# Rare-level audit for high-cardinality categoricals: define your "rare" threshold and propose grouping into "Other".
def rare_level_audit(df, cardinality_df, rare_threshold=0.01):

    audit_rows = []

    high_cols = cardinality_df.loc[
        cardinality_df["cardinality_level"] == "High",
        "column"
    ]

    for col in high_cols:

        if col not in df.columns:
            continue

        counts = df[col].value_counts(dropna=False)
        freqs = counts / len(df)

        rare_levels = freqs[freqs < rare_threshold]

        for level in rare_levels.index:
            audit_rows.append({
                "column": col,
                "category": str(level),
                "count": int(counts[level]),
                "frequency_pct": round(freqs[level] * 100, 3),
                "recommendation": "Group into 'Other'"
            })

    audit_df = pd.DataFrame(audit_rows)

    if not audit_df.empty:
        audit_df = audit_df.sort_values(
            ["column", "frequency_pct"]
        ).reset_index(drop=True)

    return audit_df

for i in range(len(datasets)):
    result = rare_level_audit(datasets[i], cardinality_df)
    if i == 10:
        print(f'Rare level audit completed for CT911 dataset')
    elif i==11:
        print(f'Rare level audit completed for Parcel_CAMA dataset')
    else:
        print(f'Rare level audit completed for dataset {i+1}')
    print(result)
    print()

In [ ]:
# categorical masquerading as numeric
def classify_numeric_role(df):
    results = []

    for col in df.select_dtypes(include=np.number).columns:

        s = df[col].dropna()

        if len(s) == 0:
            continue

        unique_ratio = s.nunique() / len(s)

        if np.allclose(s, np.round(s)):

            if s.nunique() <= 20:
                role = "Categorical Encoding"

            elif unique_ratio < 0.05:
                role = "Possible Code/ZIP"

            else:
                role = "Numeric"

        else:
            role = "Numeric"

        results.append({
            "column": col,
            "role": role,
            "unique_values": s.nunique(),
            "unique_ratio": round(unique_ratio, 4)
        })

    return pd.DataFrame(results)

for i in range(len(datasets)):
    df = datasets[i]
    result = classify_numeric_role(df)
    print(f"Dataset {i+1} results:")
    print(result)
    print()

### Stage 5 — Interpretation

*Which features need transformation? Which categoricals need rare-level grouping? Any "numeric-looking" categoricals you reclassified?*


The columns that were recommended to have a transformation were columns that it wouldn't make sense to transform in the first place. For instance, the ones on the graphs that showed the best distribution after the transformations were addresss street number but that variable is not looking at a continuous distribution. Rather it is giving a discripter of the overall address that accompanies several other variables in each of the datasets. The only categorical variable that would make sense to need rare-level grouping would be the municipality or towns as there are a fixed number of towns in the dataset. However, there are over 100 towns within the dataset and having a categorical variable with over 100 levels isn't practical or feasible for computations. There are several instances where zip codes came in under different data types, including number, string, and categorical. Going further we need to make sure that all of the zip codes populate with the same numeric data type and contain any missing leading zeros.

## Stage 6 — Target Analysis *(10 pts)*
**Classification target:**
- Class frequencies and proportions; bar chart.
- Imbalance ratio.
- Recommended response (none / class weights / resampling / threshold tuning), tied to the metric chosen in Stage 1.
- Stratification implications for your Stage 10 split.

**Regression target:**
- Distribution, summary statistics, skew, kurtosis.
- Outlier and zero-inflation check.
- Transformation evaluation (e.g., log-target) with before/after plots; recommend whether to model on the transformed scale.
- Identify any categoricals masquerading as numeric (e.g., zip codes).

**Both:**
- If a temporal axis exists, plot target over time and comment on drift, seasonality, or regime change.

In [ ]:
# Target distribution
def plot_relative_frequency(target_col):
    cleaned = target_col.astype(str).str.strip().str.title()
    rel_freq = cleaned.value_counts(normalize=True, dropna=False)

    plt.figure(figsize=(10, 6))
    rel_freq.sort_values(ascending=False).plot(kind='bar')
    plt.title(f"Relative Frequency Distribution: {cleaned.name}")
    plt.xlabel("Town")
    plt.ylabel("Relative Frequency")
    plt.xticks(rotation=90, ha='right', fontsize = 8)
    plt.tight_layout()
    plt.show()

    return rel_freq.reset_index().rename(
        columns={"index": "value", cleaned.name: "relative_frequency"}
    )

for i in range(len(datasets)):
    if i <= 4:
        target_col = datasets[i]["MNCPLT_NM"]
    elif i == 5:
        target_col = datasets[i]["CITY_NM"]
    elif i == 6:
        target_col = datasets[i]["City"]
    elif i == 7:
        continue
    elif i == 8:
        continue
    elif i == 9:
        target_col = datasets[i]["City"]
    elif i == 10:
        target_col = datasets[i]["Inc_Muni"]
    else:
        target_col = datasets[i]["Town_Name"]
    if i == 10:
        print("Relative frequency for Towns in CT911 Dataset")
    elif i == 11:
        print("Relative frequency for Towns in Parcel_CAMA Dataset")
    else:
        print(f"Relative frequency for Towns in Dataset {i+1}")
    plot_relative_frequency(target_col)

In [ ]:
# Imbalance Ratio
def imbalance_ratio(target_col):
    cleaned = target_col.astype(str).str.strip().str.title()

    class_counts = cleaned.value_counts(dropna=False)
    smallest_class = class_counts.idxmin()
    smallest_count = class_counts.min()


    largest_class = class_counts.idxmax()
    largest_count = class_counts.max()


    ir = largest_count / smallest_count


    return pd.DataFrame({
        "smallest_class": [smallest_class],
        "smallest_count": [smallest_count],
        "largest_class": [largest_class],
        "largest_count": [largest_count],
        "imbalance_ratio": [round(ir, 4)]
    })


for i in range(len(datasets)):
    if i <= 4:
        target_col = datasets[i]["MNCPLT_NM"]
    elif i == 5:
        target_col = datasets[i]["CITY_NM"]
    elif i == 6:
        target_col = datasets[i]["City"]
    elif i == 7:
        continue
    elif i == 8:
        continue
    elif i == 9:
        target_col = datasets[i]["City"]
    elif i == 10:
        target_col = datasets[i]["Inc_Muni"]
    else:
        target_col = datasets[i]["Town_Name"]
    if i == 10:
        print("Imbalance ratio for Towns in CT911 Dataset")
    elif i == 11:
        print("Imbalance ratio for Towns in Parcel_CAMA Dataset")
    else:
        print(f"Imbalance ratio for Towns in Dataset {i+1}")
    imbalance = imbalance_ratio(target_col)
    print(imbalance)
    print()


In [ ]:
# Recommendation for class imbalance
def imbalance_strategy_recommendation(target_col):
    cleaned = target_col.astype(str).str.strip().str.title()

    counts = cleaned.value_counts(dropna=False)
    n_classes = len(counts)
    largest = counts.max()
    smallest = counts.min()
    imbalance_ratio = largest / smallest

    singleton_classes = (counts == 1).sum()
    singleton_ratio = singleton_classes / n_classes
    rare_classes = (counts <= 5).sum()
    rare_ratio = rare_classes / n_classes

    effective_classes = (counts >= 10).sum()

    print(singleton_ratio)
    print(effective_classes)
    if singleton_ratio > 0.25 or effective_classes < 5:
        recommendation = "reframe_problem"
        rationale = (
            "Too many classes have extremely low support. "
            "Model cannot reliably learn these categories. "
            "Consider grouping rare classes or redefining target."
        )

    elif imbalance_ratio > 10:
        recommendation = "class_weights + threshold_tuning"
        rationale = (
            "Moderate-to-severe imbalance. "
            "Use weighting and threshold tuning."
        )

    elif imbalance_ratio > 2:
        recommendation = "class_weights"
        rationale = "Mild imbalance."

    else:
        recommendation = "none"
        rationale = "Balanced distribution."


    return pd.DataFrame({
        "n_classes": [n_classes],
        "effective_classes_(>=10_samples)": [effective_classes],
        "imbalance_ratio": [round(imbalance_ratio, 4)],
        "singleton_ratio": [round(singleton_ratio, 4)],
        "rare_ratio_(<=5)": [round(rare_ratio, 4)],
        "recommended_strategy": [recommendation],
        "rationale": [rationale]
    })

for i in range(len(datasets)):
    if i <= 4:
        target_col = datasets[i]["MNCPLT_NM"]
    elif i == 5:
        target_col = datasets[i]["CITY_NM"]
    elif i == 6:
        target_col = datasets[i]["City"]
    elif i == 7:
        continue
    elif i == 8:
        continue
    elif i == 9:
        target_col = datasets[i]["City"]
    elif i == 10:
        target_col = datasets[i]["Inc_Muni"]
    else:
        target_col = datasets[i]["Town_Name"]
    print(f"Dataset {i+1}:")
    print(imbalance_strategy_recommendation(target_col))

In [ ]:
# Stratification implications
def stratification_implications(target_col, n_splits=5):
    cleaned = target_col.astype(str).str.strip().str.title()

    counts = cleaned.value_counts(dropna=False)

    n_classes = len(counts)
    total = counts.sum()

    largest = counts.max()
    smallest = counts.min()
    imbalance_ratio = largest / smallest

    singleton_classes = (counts == 1).sum()
    rare_classes = (counts <= n_splits).sum()

    singleton_ratio = singleton_classes / n_classes
    rare_ratio = rare_classes / n_classes

    min_class_size = counts.min()

    if min_class_size < 2:
        feasibility = "invalid"
        implication = (
            "Stratification not possible: at least one class has only 1 sample."
        )

    elif min_class_size < n_splits:
        feasibility = "unstable"
        implication = (
            "Some classes cannot appear in all folds. "
            "Expect high variance in validation scores."
        )

    elif rare_ratio > 0.3:
        feasibility = "risky"
        implication = (
            "Many low-frequency classes. "
            "Some folds will have missing classes."
        )

    elif imbalance_ratio > 50:
        feasibility = "unstable"
        implication = (
            "Extreme imbalance may cause uneven fold representation."
        )

    else:
        feasibility = "safe"
        implication = "Stratified splitting is appropriate."

    return pd.DataFrame({
        "n_classes": [n_classes],
        "total_samples": [total],
        "imbalance_ratio": [round(imbalance_ratio, 4)],
        "min_class_size": [min_class_size],
        "singleton_ratio": [round(singleton_ratio, 4)],
        "rare_ratio_(<=k_splits)": [round(rare_ratio, 4)],
        "feasibility": [feasibility],
        "implication": [implication]
    })

for i in range(len(datasets)):
    if i <= 4:
        target_col = datasets[i]["MNCPLT_NM"]
    elif i == 5:
        target_col = datasets[i]["CITY_NM"]
    elif i == 6:
        target_col = datasets[i]["City"]
    elif i == 7:
        continue
    elif i == 8:
        continue
    elif i == 9:
        target_col = datasets[i]["City"]
    elif i == 10:
        target_col = datasets[i]["Inc_Muni"]
    else:
        target_col = datasets[i]["Town_Name"]
    print(f"Dataset {i+1}:")
    print(stratification_implications(target_col))

In [ ]:
# Skewness and Kurtosis
for i in range(len(datasets)):
    df = datasets[i]
    numeric_df = keep_only_numeric_like_columns(df)
    summary = summary_stats(numeric_df)
    if i == 10:
        print(f'Summary stats for CT911 Dataset:')
    elif i == 11:
        print(f'Summary stats for Parcel_CAMA Dataset:')
    else:
        print(f'Summary stats for Dataset {i+1}:')
    print(summary)

In [ ]:
# outliers and zero-inflation check
def numeric_outlier_and_zero_inflation(df, iqr_multiplier=1.5, zero_threshold=0.2):
    numeric_df = df.select_dtypes(include=[np.number])

    results = []

    for col in numeric_df.columns:
        series = numeric_df[col].dropna()

        if len(series) == 0:
            continue

        # ---- Zero inflation ----
        zero_count = (series == 0).sum()
        zero_ratio = zero_count / len(series)
        zero_inflated = zero_ratio >= zero_threshold

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - iqr_multiplier * iqr
        upper_bound = q3 + iqr_multiplier * iqr

        outlier_mask = (series < lower_bound) | (series > upper_bound)
        outlier_count = outlier_mask.sum()
        outlier_ratio = outlier_count / len(series)

        results.append({
            "column": col,
            "zero_ratio": round(zero_ratio, 4),
            "zero_inflated": zero_inflated,
            "outlier_count": int(outlier_count),
            "outlier_ratio": round(outlier_ratio, 4),
            "lower_bound": lower_bound,
            "upper_bound": upper_bound
        })

    return pd.DataFrame(results)

for i in range(len(datasets)):
    df = datasets[i]
    numeric_df = keep_only_numeric_like_columns(df)
    result = numeric_outlier_and_zero_inflation(numeric_df)
    if i == 10:
        print(f'Outliers and zero-inflation for CT911 Dataset:')
    elif i == 11:
        print(f'Outliers and zero-inflation for Parcel_CAMA Dataset:')
    else:
        print(f'Outliers and zero-inflation for Dataset {i+1}:')
    print(result)
    print()

In [ ]:
# Transformation evaluation
def transform_and_plot_skewed_features(df, skew_df, col_col = "column",skew_col = "skew", skew_threshold = 1.0):

    df_transformed = df.copy()
    results = {}

    for _, row in skew_df.iterrows():
        col = row[col_col]
        skew = row[skew_col]

        if col not in df.columns:
            continue

        if np.isnan(skew):
            continue

        if abs(skew) <= skew_threshold:
            continue

        x = df[col].dropna()

        if x.nunique() < 2:
            continue

        transformed = None
        method = None

        if (x > 0).all():
            try:
                transformed, lam = stats.boxcox(x)
                method = f"Box-Cox (lambda={lam:.3f})"
            except Exception:
                transformed = np.log1p(x)
                method = "log1p"

        elif (x >= 0).all():
            transformed = np.log1p(x)
            method = "log1p"

        else:
            pt = PowerTransformer(method="yeo-johnson", standardize=False)
            transformed = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
            method = "Yeo-Johnson"

        df_transformed.loc[x.index, col] = transformed

        transformed_skew = pd.Series(transformed).skew()

        skew_improvement = abs(skew) - abs(transformed_skew)

        if abs(skew) < 1:
            recommendation = "No Transformation Needed"

        elif skew_improvement <= 0:
            recommendation = "Do Not Transform"

        elif skew_improvement < 0.5:
            recommendation = "Optional Transformation"

        else:
            recommendation = "Recommend Transformation"

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        axes[0].hist(x, bins=30, edgecolor="black")
        axes[0].set_title(f"Before: {col}\nskew={skew:.2f}")

        axes[1].hist(transformed, bins=30, edgecolor="black")
        axes[1].set_title(f"After: {col}\n{method}\nskew={transformed_skew:.2f}")

        plt.tight_layout()
        plt.show()

        results[col] = {
            "original_skew": round(skew, 4),
            "transformed_skew": round(transformed_skew, 4),
            "skew_improvement": round(skew_improvement, 4),
            "method": method,
            "recommendation": recommendation
        }

    return df_transformed, pd.DataFrame.from_dict(results, orient="index")

for i in range(len(datasets)):
    df = datasets[i]
    numeric_df = keep_only_numeric_like_columns(df)
    summary = summary_stats(numeric_df)
    if i == 10:
        print(f'Transformations for CT911 dataset')
    elif i == 11:
        print(f'Transformations for Parcel_CAMA dataset')
    else:    
        print(f'Transformations for dataset {i+1}')
    results = transform_and_plot_skewed_features(numeric_df, summary)
    display(results)

### Stage 6 — Interpretation

*State your recommended response (class weights / resampling / target transform / etc.) and tie it back to the metric chosen in Stage 1.*

Transformations do not make sense for Street/Address Number, Zipcodes,  or Longitude/Latitude variables. The only ones it would make sense to actually adjust would be the land acres as those would be just to get simple statistics not in confirming the validity of an address itself. There are a few ID columns in dataset 6 that are calculated to require transformations. However, this follows the same thought process as Street/Address number as that is an identifier rather than any information about the validity of the address itself. There are not any varaibles that would make sense to transform as they all provide information specific to the address itself which needs to be maintained in order to judge the validity of the address.


---

## Stage 7 — Bivariate Signals vs. Target  *(12 pts)*

For **every retained feature**, quantify its bivariate relationship with the target using a test appropriate to the variable-type pair.

| Feature type | Classification target | Regression target |
|---|---|---|
| Continuous / count | Point-biserial *r*; ANOVA F or Kruskal-Wallis H; AUC of feature alone | Pearson *r*, Spearman *ρ* (report both) |
| Ordinal | Mann-Whitney / Kruskal-Wallis; Spearman *ρ* | Spearman *ρ*, Kendall *τ* |
| Nominal | Chi-square + **Cramér's V**; target rate by level | One-way ANOVA + **η²**; group means with CIs |
| Binary | Two-proportion z; difference in target rate | t-test or Mann-Whitney; **Cohen's *d*** |

**Required output:**
- A ranked table of all features with: test used, statistic, p-value, **effect size**, one-line interpretation.
- At least 6 illustrative plots (violin/box-by-class, target-rate-by-bin, scatter with LOWESS).
- Explicit acknowledgement of multiple-comparison risk (Bonferroni, Benjamini-Hochberg, or a stated screening caveat).

> A statistically significant *p*-value with a trivial effect size is **not a signal**. Effect size is required.

In [ ]:
# Bivariate tests — numeric features vs target
# Analysis on land acreage. Only real continuous variable data contains.
def feature_target_correlations(numeric_df, target_col_name):
    target = numeric_df[target_col_name]

    results = []

    for col in numeric_df.columns:
        if col == target_col_name:
            continue

        temp = numeric_df[[col, target_col_name]].dropna()

        if len(temp) < 3:
            continue

        try:
            pearson_r, pearson_p = pearsonr(temp[col], temp[target_col_name])
        except Exception:
            pearson_r, pearson_p = np.nan, np.nan

        try:
            spearman_rho, spearman_p = spearmanr(
                temp[col], temp[target_col_name]
            )
        except Exception:
            spearman_rho, spearman_p = np.nan, np.nan

        results.append({
            "feature": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p
        })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(
            by="pearson_r",
            key=lambda x: x.abs(),
            ascending=False
        ).reset_index(drop=True)

    return results_df



numeric_parcel_CAMA = keep_only_numeric_like_columns(parcel_CAMA)
correlations = feature_target_correlations(numeric_parcel_CAMA, 'Land_Acres')
print(correlations)

In [ ]:
# Nominal vs Continuous target
# Size of the plot of land based on the city that it is in
def town_anova_report(df, town_col, target_col):

    data = df[[town_col, target_col]].dropna()

    groups = [
        g[target_col].values
        for _, g in data.groupby(town_col)
    ]

    # ANOVA
    F_stat, p_val = f_oneway(*groups)

    # Grand mean
    y = data[target_col].values
    grand_mean = np.mean(y)

    # SS components
    ss_between = sum(
        len(g) * (np.mean(g) - grand_mean) ** 2
        for g in groups
    )
    ss_total = np.sum((y - grand_mean) ** 2)

    eta_sq = ss_between / ss_total if ss_total > 0 else np.nan

    def effect(x):
        if x >= 0.14:
            return "Large town effect"
        elif x >= 0.06:
            return "Moderate town effect"
        elif x >= 0.01:
            return "Small town effect"
        else:
            return "Negligible town effect"

    interpretation = effect(eta_sq)


    return pd.DataFrame([{
        "Feature": town_col,
        "Test Used": "One-way ANOVA",
        "Statistic": F_stat,
        "p-value": p_val,
        "Effect Size": eta_sq,
        "Interpretation": interpretation
    }])

anova_results = town_anova_report(parcel_CAMA, 'Town_Name', 'Land_Acres')
print(anova_results)

# Significant effect but weak effect size

In [ ]:
# Ranked feature table: feature | test | statistic | p | effect size | interpretation
def rank_feature_relationships_r2(df, target_col):

    results = []

    for col in df.columns:

        if col == target_col:
            continue

        temp = df[[col, target_col]].dropna()

        if len(temp) < 3:
            continue

        x = temp[col].values
        y = temp[target_col].values

        # Pearson test
        try:
            p_r, p_p = pearsonr(x, y)
        except:
            p_r, p_p = np.nan, np.nan

        # Spearman test
        try:
            s_rho, s_p = spearmanr(x, y)
        except:
            s_rho, s_p = np.nan, np.nan

        # Choose stronger relationship
        if abs(s_rho) > abs(p_r):
            test_used = "Spearman"
            statistic = s_rho
            p_value = s_p
            effect_size = s_rho ** 2   # rank-based pseudo R²
        else:
            test_used = "Pearson"
            statistic = p_r
            p_value = p_p
            effect_size = p_r ** 2     # true univariate linear R²

        direction = "positive" if statistic > 0 else "negative"

        interpretation = (
            f"{direction.capitalize()} relationship "
            f"explaining {effect_size:.3f} of variance in target"
        )

        results.append({
            "Feature": col,
            "Test Used": test_used,
            "Test Statistic": statistic,
            "P-value": p_value,
            "Effect Size": effect_size,
            "One-line Interpretation": interpretation
        })

    results_df = pd.DataFrame(results)

    results_df = (
        results_df
        .sort_values("Effect Size", ascending=False)
        .reset_index(drop=True)
    )

    return results_df


numeric_parcel_CAMA = keep_only_numeric_like_columns(parcel_CAMA)
ranked_features = rank_feature_relationships_r2(numeric_parcel_CAMA, 'Land_Acres')
print(ranked_features)

In [ ]:
# Unified Ranked feature table: feature | test | statistic | p | effect size | interpretation
def combine_feature_reports(df, target_col, cat_col):
    numeric_df = keep_only_numeric_like_columns(df)
    numeric_results = rank_feature_relationships_r2(numeric_df, target_col)
    categorical_results = town_anova_report(df, cat_col, target_col)

    numeric_results = numeric_results.rename(columns={
        "P-value": "p-value"
    })

    categorical_results = categorical_results.rename(columns={
        "Feature": "Feature",
        "Test Used": "Test Used",
        "Statistic": "Statistic",
        "p-value": "p-value",
        "Effect Size": "Effect Size",
        "Interpretation": "Interpretation"
    })

    combined = pd.concat(
        [numeric_results, categorical_results],
        ignore_index=True
    )

    if combined.empty:
        return combined

    def extract_effect(x):
        if isinstance(x, str):
            try:
                return float(x.split("=")[-1])
            except:
                return 0.0
        return abs(x)

    combined["ranking_score"] = combined["Effect Size"].apply(extract_effect)

    combined = combined.sort_values(
        "ranking_score",
        ascending=False
    ).drop(columns="ranking_score")

    combined.insert(0, "Rank", range(1, len(combined) + 1))

    return combined.reset_index(drop=True)

combined_results = combine_feature_reports(parcel_CAMA, 'Land_Acres', 'Town_Name')
print(combined_results)

In [ ]:
# Illustrative plots (6+) with takeaway captions
plt.figure(figsize=(10, 8))
sns.barplot(
    data=ranked_features,
    y="Feature",
    x="Effect Size"
)

plt.title(f"Top Features by Effect Size")
plt.tight_layout()
plt.show()

# Features at the top exhibit the strongest relationship with land acreage and should receive the greatest attention during feature engineering and modeling.

In [ ]:
# Another plot
top_towns = (
    parcel_CAMA["Town_Name"]
    .value_counts()
    .head(15)
    .index
)

plot_df = parcel_CAMA[parcel_CAMA["Town_Name"].isin(top_towns)]

plt.figure(figsize=(14,6))

sns.boxplot(
    data=plot_df,
    x="Town_Name",
    y="Land_Acres"
)

plt.xticks(rotation=45)
plt.title("Distribution of Land Acres by Town")
plt.tight_layout()
plt.show()

# Large differences in median and spread between towns indicate that location is a meaningful driver of parcel size.

In [ ]:
# Town Mean Acres Plot

town_summary = (
    parcel_CAMA.groupby("Town_Name")["Land_Acres"]
      .mean()
      .sort_values(ascending=False)
      .head(20)
      .iloc[2:]
      .reset_index()
)

plt.figure(figsize=(12,6))

sns.barplot(
    data=town_summary,
    x="Town_Name",
    y="Land_Acres"
)

plt.xticks(rotation=45)
plt.title("Average Land Acres by Town")
plt.tight_layout()
plt.show()

# Differences in average parcel size across towns help explain the ANOVA effect size and identify locations with systematically larger or smaller properties.

In [ ]:
# Sample Size vs mean Land Acres

town_summary = (
    parcel_CAMA.groupby("Town_Name")
      .agg(
          mean_acres=("Land_Acres", "mean"),
          n=("Land_Acres", "size")
      )
      .reset_index()
)
town_summary = town_summary[town_summary["Town_Name"] != "Sherman"]
town_summary = town_summary[town_summary["Town_Name"] != "Eastford"]

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=town_summary,
    x="n",
    y="mean_acres"
)

plt.xlabel("Number of Parcels")
plt.ylabel("Average Land Acres")
plt.title("Town Sample Size vs Average Parcel Size")

plt.tight_layout()
plt.show()

# Towns with unusually large average parcel sizes but very small sample counts should be interpreted cautiously, as a few large properties may disproportionately influence the ANOVA results.

In [ ]:
# Distribution of Land Acres
plot_data = np.log1p(
    parcel_CAMA.loc[parcel_CAMA["Land_Acres"] <= 100, "Land_Acres"]
)

plt.figure(figsize=(10, 6))
plt.hist(plot_data, bins=50)

plt.title("Log-Transformed Distribution of Land Acres (≤ 100 Acres)")
plt.xlabel("log(1 + Land Acres)")
plt.ylabel("Count")
plt.show()

# A strong right skew indicates that a small number of very large parcels may dominate statistical relationships

In [ ]:
# CDF of land Acres

plot_data = np.log1p(
    parcel_CAMA.loc[parcel_CAMA["Land_Acres"] <= 100, "Land_Acres"]
).dropna().sort_values()

# CDF values
cdf = np.arange(1, len(plot_data) + 1) / len(plot_data)

plt.figure(figsize=(10, 6))

plt.plot(plot_data, cdf)

plt.xlabel("log(1 + Land Acres)")
plt.ylabel("Cumulative Proportion")
plt.title("CDF of Land Acres (Log-Transformed)")

plt.grid(alpha=0.3)

plt.show()

# The curve shows the proportion of parcels below any given acreage threshold. Steeper sections indicate where most properties are concentrated, while flatter sections reveal relatively uncommon parcel sizes.

### Stage 7 — Interpretation

*Which features show meaningful effect sizes (not just significance)? Name at least one significant-but-trivial feature and one borderline-significant-but-practically-interesting feature. Address multiple-comparison risk.*

This analysis was conducted on the Land Acres in the Parcel_CAMA dataset as that is the only continuous variable that would make sense to perform any analysis on. The most significant feature would be the Town Name. This makes sense as the town an address is located in would affect the amount of area that plot of land can have, an address in a suburban town would have more acres than an urban address. None of the variables were all too significant however, as they may not be the best predictors in land acres. Additionally, land acres is not the dependent variable of the entire project so analysis ontheres specific features are aren not the greatest descripter of the overall validity of the data.

---

## Stage 8 — Multivariate Structure  *(12 pts)*

**Produce:**
- **Pearson and Spearman** correlation matrices for numeric features (heatmaps). Interpret disagreements (linear vs. monotonic).
- Multicollinearity assessment via **VIF** (or condition number) on the candidate numeric feature set; flag VIF > 10.
- Pairplots or facetted scatter for the top 6–8 features identified in Stage 7, colored/grouped by target.
- **Multivariate outlier detection**: Isolation Forest *or* Mahalanobis distance on the numeric feature set. Investigate the top 1% — errors, edge cases, or a meaningful subpopulation?
- *Encouraged but optional:* PCA scree plot and 2-component projection colored by target.

In [ ]:
# Pearson + Spearman correlation matrices; comment on disagreements

def corr_heatmaps(num_df, figsize=(14, 6), cmap="coolwarm", annot=True):
    # Correlation matrices
    pearson_corr = num_df.corr(method="pearson")
    spearman_corr = num_df.corr(method="spearman")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    im1 = axes[0].imshow(pearson_corr.values, cmap=cmap, vmin=-1, vmax=1)
    axes[0].set_title("Pearson Correlation (Linear)")
    axes[0].set_xticks(range(len(pearson_corr.columns)))
    axes[0].set_yticks(range(len(pearson_corr.columns)))
    axes[0].set_xticklabels(pearson_corr.columns, rotation=90)
    axes[0].set_yticklabels(pearson_corr.columns)

    im2 = axes[1].imshow(spearman_corr.values, cmap=cmap, vmin=-1, vmax=1)
    axes[1].set_title("Spearman Correlation (Monotonic)")
    axes[1].set_xticks(range(len(spearman_corr.columns)))
    axes[1].set_yticks(range(len(spearman_corr.columns)))
    axes[1].set_xticklabels(spearman_corr.columns, rotation=90)
    axes[1].set_yticklabels(spearman_corr.columns)

    fig.colorbar(im1, ax=axes, fraction=0.02, pad=0.02)
    fig.legend(
        ["Pearson", "Spearman"],
        loc="center",
        bbox_to_anchor=(0.5, 0.5),
        frameon=False
    )

    plt.show()

    return pearson_corr, spearman_corr

numeric_parcel_CAMA = keep_only_numeric_like_columns(parcel_CAMA)
pearson_corr, spearman_corr = corr_heatmaps(numeric_parcel_CAMA)
print(f"Pearson Correlation: {pearson_corr}")
print()
print(f"Spearman Correlation: {spearman_corr}")

# In a Pearson/Spearman correlation heatmap using a diverging colormap (like coolwarm), colors typically represent the strength and direction of correlation, where warm colors (e.g., red) 
# indicate strong positive correlation and cool colors (e.g., blue) indicate strong negative correlation. Values near the middle (often white or light tones) indicate weak or no correlation, 
# meaning the variables do not move together in a consistent linear or monotonic way.

In [ ]:
# VIF / multicollinearity assessment
def vif_assessment(df, threshold=10.0):

    X = df.select_dtypes(include=[np.number]).dropna()

    X = X.copy()
    X["intercept"] = 1

    vif_data = []

    for i in range(X.shape[1]):
        vif_value = variance_inflation_factor(X.values, i)
        vif_data.append((X.columns[i], vif_value))

    vif_df = pd.DataFrame(vif_data, columns=["feature", "VIF"])

    # Remove intercept row from output
    vif_df = vif_df[vif_df["feature"] != "intercept"]

    # Flag high multicollinearity
    vif_df["flag_multicollinearity"] = vif_df["VIF"] > threshold

    return vif_df.sort_values("VIF", ascending=False)

parcel_vif_assessment = vif_assessment(parcel_CAMA)
print(parcel_vif_assessment)

# The VIF results assess the degree of multicollinearity among the candidate numeric features by quantifying how well each feature can be linearly explained by the others. 
# The feature with the highest VIF was Property_Zip, indicating it has the strongest linear dependency with the remaining predictors, although its magnitude did not exceed the critical threshold for concern. 
# Importantly, none of the primary features were labeled as flagged (VIF > 10), suggesting that multicollinearity is not severe enough to compromise model stability or inflate coefficient estimates in the current feature set.

In [ ]:
# Multivariate outlier detection:
def multivariate_outlier_detection(df, method="isolation_forest", contamination=0.01, random_state=42):

    X = df.select_dtypes(include=np.number).dropna()

    if X.empty:
        raise ValueError("No numeric data available.")

    if method.lower() == "isolation_forest":

        model = IsolationForest(
            contamination=contamination,
            random_state=random_state
        )

        model.fit(X)

        scores = -model.score_samples(X)
        labels = model.predict(X)

    elif method.lower() == "mahalanobis":

        covariance = np.cov(X.T)

        inv_cov = np.linalg.pinv(covariance)

        mean = X.mean().values

        scores = np.array([
            mahalanobis(row, mean, inv_cov)
            for row in X.values
        ])

        threshold = np.quantile(scores, 1 - contamination)

        labels = np.where(scores >= threshold, -1, 1)

    else:
        raise ValueError(
            "method must be 'isolation_forest' or 'mahalanobis'"
        )

    results = X.copy()

    results["Outlier Score"] = scores
    results["Outlier"] = labels == -1

    results = results.sort_values(
        "Outlier Score",
        ascending=False
    )

    outliers = results[results["Outlier"]].copy()

    return outliers

outliers = multivariate_outlier_detection(parcel_CAMA, method="isolation_forest", contamination=0.01)
print(outliers)

### Stage 8 — Interpretation

*Where do Pearson and Spearman disagree, and what does that imply? Which VIF clusters need consolidation? What did the multivariate outliers turn out to be?*

For the most part Pearson and Spearman agree with one another in terms of directionality. There are some pairings that have positive Pearson values and negative Spearman values. This means that they are going to be some outliers in the data. There are going to be outliers within the Parcel CAMA dataset that don't follow the pattern of the other data points. There were no variables that were flagged for having multicollinearity. The highest VIF that was tested was 3.044 for the property zipcode variable. While it makes sense that zipcode doesn't have collinearity with other variables, this should cause a little more of a review when looking at how zip codes are imported into the data and read from there.

---

## Stage 9 — Mutual Information & Leakage Screen  *(12 pts)*

### 9a. Mutual Information
- Compute MI between every feature and the target using the **correct** sklearn estimator (`mutual_info_classif` for classification, `mutual_info_regression` for regression).
- Encode categoricals appropriately and pass `discrete_features` correctly.
- Rank features by MI; produce a horizontal bar chart of the top 20.
- Compare the MI ranking with Stage 7's linear-association ranking. **Identify and discuss ≥ 3 features where the two rankings disagree** — this is where non-linear signal lives.

### 9b. Leakage Screen (graded heavily)
For every feature in the top quartile of MI *or* top quartile of bivariate effect size, answer **in writing** for each feature:
1. Is this feature available **at the time of prediction**, or only after the target is realized?
2. Is it derived from, or a near-restatement of, the target?
3. Is it an identifier, timestamp, or post-event flag?
4. Does it look "too good to be true" given domain context?

Any feature failing 1–4 must be flagged **leakage-suspect** in your Data Card and excluded from the modeling set.

### 9c. Tree-based Sanity Check (Optional)
Train a single tree-based "EDA model" (e.g., `RandomForestClassifier`/`RandomForestRegressor`) **purely as an importance probe**. Report **permutation importance** on a held-out split. State the caveat: this is *not* model selection.

In [ ]:
# 9a — Mutual information (correct estimator, encoded categoricals, discrete_features mask)
def mutual_information_scores(X, problem_type, discrete_feature = "auto", random_state=42, plot=False):
    X = X.dropna()
    y = X["Land_Acres"]
    X = X.drop("Land_Acres", axis=1)

    # Compute MI
    if problem_type == "classification":
        mi = mutual_info_classif(
            X,
            y,
            discrete_features=discrete_feature,
            random_state=random_state
        )
    elif problem_type == "regression":
        mi = mutual_info_regression(
            X,
            y,
            discrete_features=discrete_feature,
            random_state=random_state
        )
    else:
        raise ValueError(
            "problem_type must be 'auto', 'classification', or 'regression'"
        )

    results = (
        pd.DataFrame({
            "feature": X.columns,
            "mutual_information": mi
        })
        .sort_values("mutual_information", ascending=False)
        .reset_index(drop=True)
    )

    return results

numeric_parcel_CAMA = keep_only_numeric_like_columns(parcel_CAMA)
results = mutual_information_scores(numeric_parcel_CAMA, problem_type="regression", plot=True)
plot=True
if plot:
        plt.figure(figsize=(8, max(4, len(results) * 0.3)))
        plt.barh(
            results["feature"][::-1],
            results["mutual_information"][::-1]
        )
        plt.xlabel("Mutual Information")
        plt.ylabel("Feature")
        plt.title(
            f"Mutual Information Scores ({"Regression".capitalize()})"
        )
        plt.tight_layout()
        plt.show()


print(results)

In [ ]:
# 9a — Top-20 MI bar chart + comparison vs Stage 7 linear ranking
def compare_mi_vs_linear(X, problem_type="auto", rankings=None, top_n=10, random_state=42, plot=True, ):
    X = X.dropna()
    y = X["Land_Acres"]
    X = X.drop("Land_Acres", axis=1)

    if problem_type == "auto":
        if (
            y.dtype == "object"
            or str(y.dtype) == "category"
            or y.nunique() <= 20
        ):
            problem_type = "classification"
        else:
            problem_type = "regression"

    corr_scores = []

    for col in X.columns:
        try:
            corr = X[col].corr(y)
        except Exception:
            corr = np.nan

        corr_scores.append(abs(corr))

    corr_df = pd.DataFrame({
        "Feature": X.columns,
        "abs_correlation": corr_scores
    })

    if problem_type == "classification":
        mi_scores = mutual_info_classif(
            X,
            y,
            random_state=random_state
        )
    else:
        mi_scores = mutual_info_regression(
            X,
            y,
            random_state=random_state
        )

    mi_df = pd.DataFrame({
        "Feature": X.columns,
        "mutual_information": mi_scores
    })

    comparison = rankings.merge(
        mi_df,
        on="Feature"
    )

    comparison["mi_rank"] = (
        comparison["mutual_information"]
        .rank(ascending=False, method="dense")
    )
    
    comparison["Rank"] = comparison["Rank"] - 1

    comparison["rank_difference"] = (
        comparison["Rank"]
        - comparison["mi_rank"]
    )

    comparison = comparison.sort_values(
        "Rank",
        ascending=True
    )
    

    if plot:

        top_features = comparison.head(top_n)

        plt.figure(figsize=(10, 6))

        plt.barh(
            top_features["Feature"][::-1],
            top_features["rank_difference"][::-1]
        )

        plt.xlabel(
            "Correlation Rank - MI Rank"
        )

        plt.ylabel("Feature")

        plt.title(
            "Features Ranked Higher by Mutual Information\n"
            "(Potential Nonlinear Signal)"
        )

        plt.tight_layout()
        plt.show()

    return comparison

rankings = combine_feature_reports(parcel_CAMA, 'Land_Acres', 'Town_Name')
comparison = compare_mi_vs_linear(numeric_parcel_CAMA, problem_type="regression", rankings=rankings, top_n=10, random_state=42, plot=True)
print(comparison)

In [ ]:
# Leakage Screen (per-feature)
def feature_leakage_audit(df):
    target_keywords = [
        "target", "label", "class", "response", "outcome",
        "actual", "truth", "default", "churn", "fraud", "acre"
    ]

    id_keywords = [
        "id", "key", "guid", "uuid", "index",
        "timestamp", "time", "date", "datetime",
        "created", "updated", "closed", "completed", "year"
    ]

    post_event_keywords = [
        "after", "post", "resolved", "closed",
        "completed", "paid", "approved",
        "cancelled", "final", "actual"
    ]

    suspicious_keywords = [
        "pred", "prediction", "score", "probability",
        "risk", "forecast", "estimate",
        "model", "output"
    ]

    rows = []

    for feature in df.columns:

        name = feature.lower()

        q1 = (
            "Target"
            if any(k in name for k in target_keywords)
            else "Known at Prediction Time"
        )

        q2 = (
            "Target"
            if any(k in name for k in target_keywords)
            else "Not Derived from Target"
        )


        q3 = (
            "This is an ID, year, or post-event feature"
            if (
                any(k in name for k in id_keywords)
                or any(k in name for k in post_event_keywords)
            )
            else "Not an ID, year, or post-event feature"
        )

        q4 = (
            "Possibly"
            if any(k in name for k in suspicious_keywords)
            else "No"
        )

        if q2 == "Target":
            decision = "Drop"
        elif q3 == "This is an ID, year, or post-event feature" or q4 == "Possibly":
            decision = "Flag"
        else:
            decision = "Keep"

        if decision == "Drop":
            decision = "Flag"

        rows.append({
            "Feature": feature,
            "Q1 (Available at prediction time?)": q1,
            "Q2 (Target-derived?)": q2,
            "Q3 (ID/timestamp/post-event?)": q3,
            "Q4 (Too good to be true?)": q4,
            "Decision": decision
        })

    return pd.DataFrame(rows)

leakage_audit = feature_leakage_audit(numeric_parcel_CAMA)
print(leakage_audit)


### 9b — Leakage Screen (per-feature)

*For every top-quartile feature, answer the four leakage questions and decide: keep / flag / drop.*

| Feature | Q1 (available at prediction time?) | Q2 (target-derived?) | Q3 (ID/timestamp/post-event?) | Q4 (too good to be true?) | Decision |
|---|---|---|---|---|---|
|  |  |  |  |  |  |


### Stage 9 — Interpretation

*Where do MI and linear correlation disagree, and what is the non-linear story? Which features did you flag as leakage-suspect, and on what evidence? Does the permutation-importance ranking support or challenge your MI ranking?*

There wasn't too many numeric variables that were flagged during the leakage screening. Collection_year and ORIG_FID are columns that are either a year or an ID. This means that they are known at the start of prediction but don't have anything to do with the prediction itself. When looking at the features themselves, Starte_Use should also be flagged as that is the same for every entry as we are only looking at the state of Connecticut. The longitude, latitude and property_zip features should absolutely be kept as those are the features that make the most sense at predicting land acres.

---

## Stage 10 — SL-Readiness Deliverables  *(15 pts)*

Produce **all four** sub-deliverables. The Data Card (10a) should also be exported as a standalone `<lastname>_<firstname>_data_card.md`.

### 10a — Data Card / Feature Dictionary

One row per feature. Internally consistent with Stages 2–9.

| Feature | Semantic type | % missing | Univariate notes | Bivariate signal (stat, effect size) | MI | Multicoll. flag | Leakage flag | Recommended treatment | Encoding | Scaling | Keep / Drop |
|---|---|---|---|---|---|---|---|---|---|---|---|
|  |  |  |  |  |  |  |  |  |  |  |  |

In [ ]:
feature_names = []
semantic_type_col = []
missing_pct_col = []
univariate_notes_col = []
bivariate_stat_col = []
bivariate_effect_col = []
mi_col = []
multicoll_flag_col = []
leakage_flag_col = []
treatment_col = []
encoding_col = []
scaling_col = []
keep_drop_col = []

for i in range(len(datasets)):
    df = datasets[i]
    print(f"Processing dataset {i + 1}")
    feature_names = feature_names + df.columns.tolist()
    print(feature_names)
    semantic_type_col = semantic_type_col + create_feature_table(df)["semantic type"].tolist()

    v_matrix = missingness_cramers_v_matrix(df)
    missing_pct_col = missing_pct_col + build_missingness_report(df, v_matrix)["missing_pct"].tolist()

    univariate_notes_col = univariate_notes_col + create_cardinality_table(df)["cardinality_ratio"].tolist()

    if i <= 10:
        for j in range(len(df.columns)):
            bivariate_stat_col.append(np.nan)
            bivariate_effect_col.append(np.nan * len(df.columns))
            mi_col.append(np.nan * len(df.columns))
            multicoll_flag_col.append(np.nan * len(df.columns))
            leakage_flag_col.append(np.nan * len(df.columns))
    elif i == 11:

        bi_var_stat = combine_feature_reports(parcel_CAMA, 'Land_Acres', 'Town_Name')
        bi_var_features = bi_var_stat["Feature"].tolist()
        feat_ind = 0
        for i in df.columns:    
            if i in bi_var_features:
                bivariate_stat_col.append(bi_var_stat["Statistic"][feat_ind])
                feat_ind = feat_ind + 1
            else:
                bivariate_stat_col.append(np.nan)
        
        bi_var_eff = combine_feature_reports(parcel_CAMA, 'Land_Acres', 'Town_Name')
        bi_var_eff_features = bi_var_stat["Feature"].tolist()
        feat_ind = 0
        for i in df.columns:    
            if i in bi_var_eff_features:
                bivariate_effect_col.append(bi_var_eff["Effect Size"][feat_ind])
                feat_ind = feat_ind + 1
            else:
                bivariate_effect_col.append(np.nan)

        mi_stat = mutual_information_scores(numeric_parcel_CAMA, problem_type="regression")
        mi_features = mi_stat["feature"].tolist()
        feat_ind = 0
        for i in df.columns:    
            if i in mi_features:
                mi_col.append(mi_stat["mutual_information"][feat_ind])
                feat_ind = feat_ind + 1
            else:
                bivariate_effect_col.append(np.nan)

        multicoll_stat = vif_assessment(numeric_parcel_CAMA)
        multicoll_features = multicoll_stat["feature"].tolist()
        feat_ind = 0
        for i in df.columns:    
            if i in mi_features:
                multicoll_flag_col.append(multicoll_stat["flag_multicollinearity"][feat_ind])
                feat_ind = feat_ind + 1
            else:
                bivariate_effect_col.append(np.nan)

        leakage_stat = feature_leakage_audit(numeric_parcel_CAMA)
        leakage_features = leakage_stat["Feature"].tolist()
        feat_ind = 0
        for i in df.columns:    
            if i in mi_features:
                leakage_flag_col.append(leakage_stat["Decision"][feat_ind])
                feat_ind = feat_ind + 1
            else:
                bivariate_effect_col.append(np.nan)

print(len(feature_names))
print(len(semantic_type_col))
print(len(missing_pct_col))
print(len(univariate_notes_col))
print(len(bivariate_stat_col))
print(len(bivariate_effect_col))
print(len(mi_col))
print(len(multicoll_flag_col))
print(len(leakage_flag_col))

data_card = pd.DataFrame({
    "Feature": feature_names,
    "Semantic type": semantic_type_col,
    "% missing": missing_pct_col,
    "Univariate notes": univariate_notes_col,
    "Bivariate stat": bivariate_stat_col,
    "Bivariate effect size": bivariate_effect_col,
    "MI": mi_col,
    "Multicollinearity flag": multicoll_flag_col,
    "Leakage flag": leakage_flag_col
})

data_card["Recommended treatment"] = "No Treatment"
data_card["Encoding"] = "No Encoding"
data_card["Scaling"] = "No Transformation"
data_card["Keep / Drop"] = "Keep"

print(data_card)

for i in data_card["Feature"]:
    if "zip" in i.lower() or "zip" in i.lower():
        encoding_col[feature_names.index(i)] = "Change to Numeric"

In [ ]:
with open('kavanagh_matt_data_card.md', 'w', encoding='utf-8') as f:
    f.write(df.to_markdown(index=False))

### 10b — Train / Validation / Test Split Recommendation

Choose **one** and justify against this dataset's specific risks:
- ☐ **Random**
- ☐ Stratified (by target)
- ☐ Grouped (specify grouping key: ____)
- ☐ Time-based (specify cutoff: ____)
- ☐ Nested

**Fractions:** train ___ / val ___ / test ___

**Justification (address leakage between splits, target stratification, group leakage, temporal validity):**


The best split for these datasets would be a random split of 70% train / 20% Validation / 10% Test. COmbined in these datasets we have over 1,000,000 records meaning that having a test set of over 100,000 would be plenty to verify whether our model properly categorizes each model into their correct bin. There is no need to stratify based on anything as the only feature that would make sense to stratify on would be town/municipality making the model have over 100 different levels and outputs. Computationally this doesn't make sense. While this model will need to be updated each year as more and more new developments come about, having the baseline features would make it easier to simply add the new addresses and pperform the validation. Based on the leakage analysis and the nature of the model itself, there were no features that were derived from the target. Our target is to classify addresses rather than specific numeric data. This means that the end goal will be to classify whether or not an address is valid/invalid/or needs review.

### 10c — Preprocessing Pipeline Sketch

Map each kept feature to its transformer chain. Specific enough that someone else could implement it as a `sklearn.pipeline.Pipeline` / `ColumnTransformer`. Mark fit-on-train-only operations clearly.

| Feature | Imputation | Transform | Encoding | Scaling | Notes (fit-on-train-only?) |
|---|---|---|---|---|---|
|  |  |  |  |  |  |

In [ ]:
# Preprocessing pipeline

preprocessing_pipeline = pd.DataFrame({
    "Feature": data_card["Feature"],
    "Imputation": treatment_col,
    "Encoding": encoding_col,
    "Scaling": scaling_col,
    "Notes": data_card["Univariate notes"],
})

### 10d — Top-10 Candidate Predictors + Known Risks

**Top 10 (with evidence cited from Stages 7 + 9 + 8):**
1. Property Zip
2. Municipality/Town Name
3. Street Name
4. Address Numver
5. 
6. 
7. 
8. 
9. 
10. Land Acres 

**Known Risks register:**
- *Leakage suspects retained or removed:*
- *Multicollinearity clusters:*
- *Missingness assumptions:*
- *Drift / temporal-stability concerns:*
- *Fairness-sensitive features and proxies:*
- *Other:*

---

## Submission Checklist

- [ ] Notebook executes top-to-bottom on a fresh kernel.
- [ ] Random seed set and documented.
- [ ] All 10 stages have numbered headers.
- [ ] Data Card exported as both notebook table and standalone `.md`.
- [ ] Readiness Memo PDF (3–5 pages) addresses: problem framing, dataset health, top signals, leakage / risk register, split & preprocessing recommendation, open questions.
- [ ] Every p-value paired with an effect size.
- [ ] Every plot has title, axis labels, and a caption.
- [ ] Leakage screen explicit and per-feature for every top-quartile feature.
- [ ] Train / val / test split strategy stated and justified.
- [ ] Preprocessing pipeline sketch present.
- [ ] Files named per convention; submitted as a single zipped folder.